In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.state import StateGraph
from pydantic import BaseModel, Field
from typing import Annotated, List, Literal, Optional
from langchain_openrouter import ChatOpenRouter
from langgraph.graph.message import add_messages
from langgraph.graph import END, START
from dotenv import load_dotenv
import os
import httpx
from bs4 import BeautifulSoup

load_dotenv(override=True)

os.environ.setdefault("LANGCHAIN_TRACING_V2", os.getenv("LANGCHAIN_TRACING_V2", "false"))
os.environ.setdefault("LANGCHAIN_PROJECT", os.getenv("LANGCHAIN_PROJECT", "Web researcher"))

llm = ChatOpenRouter(
    api_key=os.getenv("OPENROUTER_API_KEY"), model="openai/gpt-4o-mini"
)

HTTP_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

action_enum = Literal["search", "open_page", "extract", "finish"]

# ---------------------------------------------------------------------------
# How many times each action is allowed before we force-finish
# ---------------------------------------------------------------------------
MAX_ACTION_COUNTS: dict[str, int] = {
    "search": 2,
    "open_page": 2,
    "extract": 2,
    "finish": 1,
}


# ---------------------------------------------------------------------------
# Pydantic models
# ---------------------------------------------------------------------------
class PlanStep(BaseModel):
    step: str
    action: action_enum
    input: str


class PlanListStep(BaseModel):
    steps: List[PlanStep]


class ExtractedData(BaseModel):
    name: str
    price: str
    source: str
    source_url: str


class AgentState(BaseModel):
    task: str
    plan: List[PlanStep] = Field(default_factory=list)
    # action_history entries are plain strings: "<action>:<detail>"
    action_history: List[str] = Field(default_factory=list)
    extracted_data: List[ExtractedData] = Field(default_factory=list)
    current_step: int = 0
    final_answer: str = ""
    next_action: Optional[action_enum] = None
    messages: Annotated[list, add_messages] = Field(default_factory=list)
    search_summary: str = ""
    pending_url: str = ""
    page_text: str = ""
    last_url: str = ""
    # True once the Yahoo Finance direct-fetch path has been attempted
    direct_fetch_attempted: bool = False


class NextAction(BaseModel):
    action: action_enum
    reasoning: str


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _plan_input_for(state: AgentState, action: action_enum) -> str:
    for ps in state.plan:
        if ps.action == action:
            return ps.input
    return state.task


def _action_counts(state: AgentState) -> dict[str, int]:
    counts: dict[str, int] = {}
    for entry in state.action_history:
        key = entry.split(":")[0]
        counts[key] = counts.get(key, 0) + 1
    return counts


def _should_force_finish(state: AgentState) -> bool:
    """Return True when we must stop looping and go to finish."""
    counts = _action_counts(state)
    for action, max_count in MAX_ACTION_COUNTS.items():
        if counts.get(action, 0) >= max_count and action != "finish":
            # Only force if we haven't finished yet
            if counts.get("finish", 0) == 0:
                return True
    # Hard cap: more than 15 total steps without finishing
    if len(state.action_history) >= 15 and counts.get("finish", 0) == 0:
        return True
    return False


def _fetch_yahoo_finance(symbol: str = "GC=F") -> tuple[str, str, str]:
    """
    Direct JSON fetch from Yahoo Finance v8 API.
    Returns (page_text_for_extract, last_url, summary_line).
    Raises on failure.
    """
    api_url = (
        f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
        "?range=1d&interval=1m"
    )
    r = httpx.get(api_url, headers=HTTP_HEADERS, follow_redirects=True, timeout=30.0)
    r.raise_for_status()
    data = r.json()
    meta = data["chart"]["result"][0]["meta"]
    price = meta.get("regularMarketPrice") or meta.get("previousClose")
    currency = meta.get("currency", "USD")
    exchange = meta.get("exchangeName", "")
    name = meta.get("shortName") or meta.get("symbol") or symbol
    canonical_url = f"https://finance.yahoo.com/quote/{symbol}"
    page_text = (
        f"Commodity: {name}\n"
        f"Price: {price} {currency}\n"
        f"Exchange: {exchange}\n"
        f"Source URL: {canonical_url}\n"
    )
    summary = f"Yahoo Finance ({symbol}): {price} {currency} on {exchange}"
    return page_text, canonical_url, summary


# ---------------------------------------------------------------------------
# Nodes
# ---------------------------------------------------------------------------
def planner_node(state: AgentState):
    if state.current_step != 0:
        return {}
    messages = [
        HumanMessage(content=state.task),
        SystemMessage(
            content=(
                "Break the task into clear actionable steps. "
                "Each step must use one of: search, open_page, extract, finish — "
                "in a sensible order (e.g. search -> open_page -> extract -> finish). "
                "Keep the plan short: 4 steps maximum."
            )
        ),
    ]
    structured = llm.with_structured_output(PlanListStep)
    response = structured.invoke(messages)
    print("Planner:", response)
    return {"plan": response.steps, "current_step": 1}


def search_node(state: AgentState):
    """
    Try DuckDuckGo first. On any failure, fall back to Yahoo Finance direct API.
    Either way we always leave a valid pending_url and search_summary.
    """
    query = _plan_input_for(state, "search")
    lines: List[str] = []
    pending_url = ""

    # --- DuckDuckGo attempt ---
    try:
        r = httpx.get(
            "https://html.duckduckgo.com/html/",
            params={"q": query},
            headers=HTTP_HEADERS,
            follow_redirects=True,
            timeout=30.0,
        )
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        first = soup.select_one("a.result__a")
        if first and first.get("href"):
            href = first["href"]
            # DuckDuckGo sometimes wraps URLs — unwrap if needed
            if "uddg=" in href:
                from urllib.parse import urlparse, parse_qs, unquote
                qs = parse_qs(urlparse(href).query)
                href = unquote(qs.get("uddg", [href])[0])
            pending_url = href
            lines.append(f"Top hit: {first.get_text(' ', strip=True)}")
            lines.append(f"URL: {pending_url}")
        snip = soup.select_one(".result__snippet")
        if snip:
            lines.append(snip.get_text(" ", strip=True))
    except Exception as e:
        lines.append(f"DuckDuckGo search failed: {e}")

    # --- Yahoo Finance fallback (always run if no URL yet, or if it's gold) ---
    gold_keywords = {"gold", "xau", "gc=f", "gold price", "gold spot"}
    is_gold_query = any(k in query.lower() for k in gold_keywords)

    if not pending_url or is_gold_query:
        try:
            page_text, yahoo_url, summary = _fetch_yahoo_finance("GC=F")
            lines.append(f"[Yahoo Finance fallback] {summary}")
            # Prefer Yahoo URL because we know we can parse its API
            pending_url = yahoo_url
        except Exception as e:
            lines.append(f"Yahoo Finance fallback failed: {e}")

    summary_text = "\n".join(lines)
    log = f"search:{query} -> {summary_text[:300]}"
    print(f"search_node | pending_url={pending_url!r}")
    return {
        "search_summary": summary_text,
        "pending_url": pending_url,
        "action_history": state.action_history + [log],
    }


def open_page_node(state: AgentState):
    """
    For Yahoo Finance quote pages, use the JSON API directly (reliable).
    For everything else, try HTTP scraping with a clear failure log.
    """
    url = (state.pending_url or "").strip() or _plan_input_for(state, "open_page").strip()

    if not url.startswith("http"):
        msg = f"open_page:failed — no valid URL (got {url!r})"
        print(msg)
        return {
            "action_history": state.action_history + [msg],
            "page_text": "",
            "last_url": "",
        }

    # --- Yahoo Finance: use API instead of scraping the JS-heavy page ---
    if "finance.yahoo.com/quote/" in url:
        symbol = url.rstrip("/").split("/")[-1]
        try:
            page_text, last_url, summary = _fetch_yahoo_finance(symbol)
            print(f"open_page_node | Yahoo API success for {symbol}")
            return {
                "page_text": page_text,
                "last_url": last_url,
                "action_history": state.action_history + [f"open_page:{last_url}"],
            }
        except Exception as e:
            msg = f"open_page:failed — Yahoo API error: {e}"
            print(msg)
            return {
                "action_history": state.action_history + [msg],
                "page_text": "",
                "last_url": "",
            }

    # --- Generic HTTP scrape ---
    try:
        r = httpx.get(url, headers=HTTP_HEADERS, follow_redirects=True, timeout=45.0)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        text = soup.get_text(" ", strip=True)[:14000]
        if len(text.strip()) < 100:
            raise ValueError("Page returned too little text (likely JS-rendered or blocked)")
        print(f"open_page_node | scraped {len(text)} chars from {url}")
        return {
            "page_text": text,
            "last_url": url,
            "action_history": state.action_history + [f"open_page:{url}"],
        }
    except Exception as e:
        # Log failure clearly so action_selector can see it and not retry
        msg = f"open_page:failed — {e}"
        print(msg)
        # Attempt Yahoo Finance fallback automatically
        try:
            page_text, last_url, summary = _fetch_yahoo_finance("GC=F")
            print("open_page_node | fell back to Yahoo Finance API")
            return {
                "page_text": page_text,
                "last_url": last_url,
                "action_history": state.action_history + [f"open_page:{last_url}"],
            }
        except Exception as e2:
            return {
                "action_history": state.action_history + [f"open_page:failed — {e} | fallback also failed: {e2}"],
                "page_text": "",
                "last_url": "",
            }


def extract_node(state: AgentState):
    """
    Extract structured data from page_text.
    If page_text is empty, log it and move on — never silently drop.
    """
    if not state.page_text.strip():
        log = "extract:skipped — page_text was empty"
        print(log)
        return {"action_history": state.action_history + [log]}

    structured = llm.with_structured_output(ExtractedData)
    messages = [
        SystemMessage(
            content=(
                "Extract structured commodity pricing facts from the text below. "
                "Fill every field. Use the page URL as source_url."
            )
        ),
        HumanMessage(
            content=(
                f"Task: {state.task}\n"
                f"Last URL: {state.last_url}\n\n"
                f"Text:\n{state.page_text[:12000]}"
            )
        ),
    ]
    row = structured.invoke(messages)
    if state.last_url and not row.source_url:
        row = row.model_copy(update={"source_url": state.last_url})

    log = f"extract:{row.name}|{row.price}"
    print(log)
    return {
        "extracted_data": state.extracted_data + [row],
        "action_history": state.action_history + [log],
    }


def finish_node(state: AgentState):
    lines = []
    for e in state.extracted_data:
        lines.append(f"- {e.name}: {e.price} (source: {e.source}, {e.source_url})")
    facts = "\n".join(lines) if lines else "(no structured rows — summarise from search notes)"

    messages = [
        SystemMessage(
            content="Write a short, helpful summary that answers the user's task using the facts below."
        ),
        HumanMessage(
            content=(
                f"Task:\n{state.task}\n\n"
                f"Facts:\n{facts}\n\n"
                f"Search notes:\n{state.search_summary[:3000]}"
            )
        ),
    ]
    out = llm.invoke(messages)
    print("finish_node | writing final answer")
    return {
        "final_answer": out.content,
        "action_history": state.action_history + ["finish"],
    }


def action_selector(state: AgentState):
    """
    Deterministic rules first, LLM only as a tiebreaker.
    This prevents the LLM from looping on a broken action indefinitely.
    """
    history = state.action_history
    counts = _action_counts(state)

    extract_attempted = counts.get("extract", 0) > 0
    finish_done = counts.get("finish", 0) > 0

    # Rule 1: Already finished — should not happen, but guard anyway
    if finish_done:
        return {"next_action": "finish"}

    # Rule 2: Force finish if we're looping or over budget
    if _should_force_finish(state):
        print("action_selector | force-finishing due to loop/budget")
        return {"next_action": "finish"}

    # Rule 3: If extract has been attempted (success or skip), go to finish
    if extract_attempted:
        return {"next_action": "finish"}

    # Rule 4: If we have page text, extract it
    if state.page_text.strip():
        return {"next_action": "extract"}

    # Rule 5: If we have a pending URL but haven't opened it yet, open it
    open_count = counts.get("open_page", 0)
    if state.pending_url and open_count < MAX_ACTION_COUNTS["open_page"]:
        # Don't re-open the same URL that already failed
        last_open_failed = any(
            "open_page:failed" in h for h in history
        )
        if not last_open_failed or open_count == 0:
            return {"next_action": "open_page"}

    # Rule 6: If we haven't searched yet, search
    search_count = counts.get("search", 0)
    if search_count < MAX_ACTION_COUNTS["search"]:
        return {"next_action": "search"}

    # Rule 7: Fallback — let LLM decide (with full context)
    plan_txt = "\n".join(
        f"{i+1}. {p.step} | {p.action} | {p.input}"
        for i, p in enumerate(state.plan)
    )
    hist_txt = "\n".join(history[-12:])

    messages = [
        SystemMessage(
            content=(
                "Choose the single next action. Rules: "
                "use search only if you have no URL yet; "
                "use open_page only if you have a URL and no page text; "
                "use extract only if you have page text; "
                "use finish if you have extracted data OR if further progress is impossible."
            )
        ),
        HumanMessage(
            content=(
                f"Task:\n{state.task}\n\nPlan:\n{plan_txt}\n\n"
                f"Action history:\n{hist_txt}\n\n"
                f"Has page text: {bool(state.page_text.strip())}\n"
                f"Pending URL: {state.pending_url or '(none)'}\n"
                f"Extracted rows: {len(state.extracted_data)}\n"
                f"Action counts: {counts}"
            )
        ),
    ]
    structured = llm.with_structured_output(NextAction)
    response = structured.invoke(messages)
    print(f"action_selector | LLM chose: {response.action} — {response.reasoning}")
    return {"next_action": response.action}


def route(state: AgentState):
    return state.next_action


# ---------------------------------------------------------------------------
# Build graph
# ---------------------------------------------------------------------------
workflow = StateGraph(AgentState)
workflow.add_node("planner", planner_node)
workflow.add_node("action_selector", action_selector)
workflow.add_node("search", search_node)
workflow.add_node("open_page", open_page_node)
workflow.add_node("extract", extract_node)
workflow.add_node("finish", finish_node)

workflow.add_edge(START, "planner")
workflow.add_edge("planner", "action_selector")

workflow.add_conditional_edges(
    "action_selector",
    route,
    {
        "search": "search",
        "open_page": "open_page",
        "extract": "extract",
        "finish": "finish",
    },
)

workflow.add_edge("search", "action_selector")
workflow.add_edge("open_page", "action_selector")
workflow.add_edge("extract", "action_selector")
workflow.add_edge("finish", END)

app = workflow.compile()


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    initial_state = AgentState(
        task="Research the current price of Gold and write a summary.",
    )
    result = app.invoke(initial_state, {"recursion_limit": 35})

    print("\n--- GENERATED PLAN ---")
    for s in result.get("plan", []):
        if isinstance(s, dict):
            print(f"  {s['action']}: {s['input']}")
        else:
            print(f"  {s.action}: {s.input}")

    print("\n--- EXTRACTED ---")
    for e in result.get("extracted_data", []):
        if isinstance(e, dict):
            print(f"  {e['name']}: {e['price']}")
        else:
            print(f"  {e.name}: {e.price}")

    print("\n--- FINAL ANSWER ---")
    print(result.get("final_answer", ""))